## Parsing CARD reference FASTA headers

Extracts the structured fields from CARD's FASTA headers into a tabular form,
so reference sequences can be joined against RGI output on ARO accession.

Two header formats exist in the CARD data download:

| File | Format |
|---|---|
| `nucleotide_fasta_*` | `gb\|ACC\|strand\|start-end\|ARO:xxxxxxx\|gene [organism]` |
| `protein_fasta_*` | `gb\|ACC\|ARO:xxxxxxx\|gene [organism]` |

Coordinates are 0-based, end-exclusive, relative to the GenBank record —
so `end - start` is the length directly, with no `+1`.

Note: `[organism]` is the source organism of that GenBank deposit, i.e. curation
provenance for one reference sequence. It is not the gene's host range.

In [2]:
import os
import re
import pandas as pd

CARD_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data'          # directory holding the unpacked CARD data
OUTPUT_CSV = os.path.join(CARD_DIR, 'card_reference_headers.csv')

FASTA_FILES = [
    'nucleotide_fasta_protein_homolog_model.fasta',
    'protein_fasta_protein_homolog_model.fasta',
    
]

In [3]:
# Anchored on the literal 'ARO:' token rather than on field position, so
# multi-word CARD gene names (e.g. 'Escherichia coli soxS with mutation
# conferring antibiotic resistance') parse correctly instead of splitting.

PAT_NT = re.compile(
    r'^(?P<source_db>[^|]+)\|(?P<accession>[^|]+)\|(?P<strand>[+-])\|'
    r'(?P<start>\d+)-(?P<end>\d+)\|ARO:(?P<aro_accession>\d+)\|(?P<rest>.*)$'
)
PAT_PROT = re.compile(
    r'^(?P<source_db>[^|]+)\|(?P<accession>[^|]+)\|ARO:(?P<aro_accession>\d+)\|(?P<rest>.*)$'
)
PAT_ORG = re.compile(r'^(?P<gene_name>.*?)\s*\[(?P<organism>[^\]]+)\]\s*$')


def parse_card_header(header):
    """Return a dict of parsed fields, or None if the header does not match."""
    match, htype = PAT_NT.match(header), 'nucleotide'
    if not match:
        match, htype = PAT_PROT.match(header), 'protein'
    if not match:
        return None

    d = match.groupdict()
    rest = d.pop('rest')

    org_match = PAT_ORG.match(rest)
    d['gene_name'] = org_match.group('gene_name') if org_match else rest.strip()
    d['organism'] = org_match.group('organism') if org_match else ''

    if htype == 'nucleotide':
        d['start'], d['end'] = int(d['start']), int(d['end'])
        d['cds_length_bp'] = d['end'] - d['start']   # end is EXCLUSIVE
    else:
        d.update(strand='', start=pd.NA, end=pd.NA, cds_length_bp=pd.NA)

    d['header_type'] = htype
    return d


def read_fasta_records(path):
    """Yield (header, sequence) pairs. Header excludes the leading '>'."""
    header, seq = None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(seq)
                header, seq = line[1:].strip(), []
            else:
                seq.append(line.strip())
        if header is not None:
            yield header, ''.join(seq)

In [4]:
COLUMNS = ['source_file', 'source_db', 'accession', 'strand', 'start', 'end',
           'cds_length_bp', 'aro_accession', 'gene_name', 'organism',
           'seq_length', 'header_type', 'raw_header']

rows, failed_headers = [], []

for filename in FASTA_FILES:
    path = os.path.join(CARD_DIR, filename)
    for header, seq in read_fasta_records(path):
        record = parse_card_header(header)
        if record is None:
            failed_headers.append((filename, header))
            continue
        record['source_file'] = filename
        record['seq_length'] = len(seq)
        record['raw_header'] = header
        rows.append(record)

card_headers = pd.DataFrame(rows)[COLUMNS]
print(f'{len(card_headers)} records parsed, {len(failed_headers)} failed')
card_headers.head()

12118 records parsed, 0 failed


,source_file,source_db,accession,strand,start,end,cds_length_bp,aro_accession,gene_name,organism,seq_length,header_type,raw_header
0,nucleotide_fasta_protein_homolog_model.fasta,gb,GQ343019.1,+,132,1023,891,3002999,CblA-1,mixed culture bacterium AX_gF3SD01_15,891,nucleotide,gb|GQ343019.1|+|132-1023|ARO:3002999|CblA-1 [m...
1,nucleotide_fasta_protein_homolog_model.fasta,gb,HQ845196.1,+,0,861,861,3001109,SHV-52,Klebsiella pneumoniae,861,nucleotide,gb|HQ845196.1|+|0-861|ARO:3001109|SHV-52 [Kleb...
2,nucleotide_fasta_protein_homolog_model.fasta,gb,AF028812.1,+,392,887,495,3002867,dfrF,Enterococcus faecalis,495,nucleotide,gb|AF028812.1|+|392-887|ARO:3002867|dfrF [Ente...
3,nucleotide_fasta_protein_homolog_model.fasta,gb,JX017365.1,+,244,1120,876,3001989,CTX-M-130,Escherichia coli,876,nucleotide,gb|JX017365.1|+|244-1120|ARO:3001989|CTX-M-130...
4,nucleotide_fasta_protein_homolog_model.fasta,gb,JN967644.1,+,0,813,813,3002356,NDM-6,Escherichia coli,813,nucleotide,gb|JN967644.1|+|0-813|ARO:3002356|NDM-6 [Esche...


### Validation

Three independent checks:

1. **Unparseable headers** — should be zero. Anything else means CARD changed
   its header convention.
2. **Coordinate span vs actual sequence length** — `cds_length_bp` comes from the
   header coordinates, `seq_length` from counting bases in the file. A mismatch
   means CARD's stated coordinates disagree with the sequence it extracted.
3. **Divisibility by 3** — a clean CDS must be. Expect failures only in the
   rRNA gene variant model file, where the sequences are not protein-coding.

In [5]:
if failed_headers:
    print(f'UNPARSEABLE ({len(failed_headers)}):')
    for fn, h in failed_headers[:5]:
        print(f'  {fn}: {h!r}')
else:
    print('All headers parsed.')

nt = card_headers[card_headers['header_type'] == 'nucleotide']

mismatch = nt[nt['cds_length_bp'] != nt['seq_length']]
print(f'\nCoordinate/sequence length mismatches: {len(mismatch)}')
if len(mismatch):
    display(mismatch[['accession', 'aro_accession', 'gene_name',
                      'cds_length_bp', 'seq_length']].head())

not_div3 = nt[nt['cds_length_bp'] % 3 != 0]
print(f'Nucleotide records not divisible by 3: {len(not_div3)}')
if len(not_div3):
    display(not_div3[['accession', 'aro_accession', 'gene_name',
                      'cds_length_bp', 'source_file']].head())

print(f'\nDistinct ARO accessions : {card_headers["aro_accession"].nunique()}')
print(f'Records with no organism: {(card_headers["organism"] == "").sum()}')
print(f'Distinct organisms      : {card_headers["organism"].nunique()}')

All headers parsed.

Coordinate/sequence length mismatches: 0
Nucleotide records not divisible by 3: 0

Distinct ARO accessions : 6059
Records with no organism: 232
Distinct organisms      : 674


In [6]:
card_headers.to_csv(OUTPUT_CSV, index=False)
print(f'-> {OUTPUT_CSV}')

# index=False avoids the trailing-comma 'Unnamed: N' columns.
# keep_default_na=False on re-read: an empty organism field, or any gene
# literally named 'nan', would otherwise come back as NaN.
check = pd.read_csv(OUTPUT_CSV, keep_default_na=False, dtype={'aro_accession': str})
print(f'Round-trip: {len(check)} rows, {len(check.columns)} columns')

-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/card_reference_headers.csv
Round-trip: 12118 rows, 13 columns


In [7]:
!dir

AMR_Gene_Family_Distribution.ipynb
ARG_Chromosome_Plasmid_Mapping.ipynb
ARG_Family_classification.ipynb
CARD_DATA
CARD_Database.ipynb
CARD_Results
extract-fasta.py
extract_proteins.sh
history_2026-07-21.txt
Mobsuite_Chromosome_Plasmid_Mobility.ipynb
Papers
Project_Summary_Hypothesis_Results_and_Next_Steps.md
README.md
Results
Untitled6.ipynb


## Sec_3

## Joining AMR Gene Family from `aro_index.tsv`

Adds an `amr_gene_family` column to `card_reference_headers.csv` by joining on ARO
accession against CARD's `aro_index.tsv` (in `refined_details/`), which maps each
ARO Accession to its AMR Gene Family (e.g. `CTX-M beta-lactamase`).

`card_reference_headers.csv`'s `aro_accession` column holds bare digits (e.g.
`3002999`); `aro_index.tsv`'s `ARO Accession` column is prefixed (e.g.
`ARO:3002999`) — normalized before the join.

A handful of ARO Accessions appear twice in `aro_index.tsv` (multiple GenBank
deposits for the same ARO entry, e.g. two accessions for CMY-106). Every such
duplicate pair carries an identical `AMR Gene Family` value, so the join uses a
deduplicated `ARO Accession -> AMR Gene Family` lookup and asserts that no ARO
Accession maps to more than one distinct family.

This section is self-contained: it reads both inputs fresh from disk rather than
relying on the `card_headers` DataFrame built earlier in this notebook, and writes
to a new output file rather than overwriting `card_reference_headers.csv`.

In [8]:
REFINED_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details'
INPUT_HEADERS_CSV = os.path.join(REFINED_DIR, 'card_reference_headers.csv')
INPUT_ARO_INDEX_TSV = os.path.join(REFINED_DIR, 'aro_index.tsv')
OUTPUT_HEADERS_WITH_FAMILY_CSV = os.path.join(REFINED_DIR, 'card_reference_headers_with_family.csv')

In [9]:
headers_df = pd.read_csv(INPUT_HEADERS_CSV, keep_default_na=False, dtype=str)
aro_index = pd.read_csv(INPUT_ARO_INDEX_TSV, sep='\t', keep_default_na=False, dtype=str)

# card_reference_headers.csv stores bare digits ('3002999'); aro_index.tsv is prefixed ('ARO:3002999')
headers_df['aro_accession_full'] = 'ARO:' + headers_df['aro_accession']

family_lookup = aro_index[['ARO Accession', 'AMR Gene Family']].drop_duplicates()
conflicting = family_lookup[family_lookup.duplicated('ARO Accession', keep=False)]
assert conflicting.empty, f'ARO Accession maps to >1 AMR Gene Family:\n{conflicting}'

family_lookup = family_lookup.set_index('ARO Accession')['AMR Gene Family']

In [10]:
headers_df['amr_gene_family'] = headers_df['aro_accession_full'].map(family_lookup)

unmatched = headers_df[headers_df['amr_gene_family'].isna()]
print(f'{len(headers_df)} rows, {len(unmatched)} with no AMR Gene Family match')
if len(unmatched):
    display(unmatched[['accession', 'aro_accession', 'gene_name']].drop_duplicates().head())

headers_df = headers_df.drop(columns='aro_accession_full')
headers_df.head()

12118 rows, 0 with no AMR Gene Family match


,source_file,source_db,accession,strand,start,end,cds_length_bp,aro_accession,gene_name,organism,seq_length,header_type,raw_header,amr_gene_family
0,nucleotide_fasta_protein_homolog_model.fasta,gb,GQ343019.1,+,132,1023,891,3002999,CblA-1,mixed culture bacterium AX_gF3SD01_15,891,nucleotide,gb|GQ343019.1|+|132-1023|ARO:3002999|CblA-1 [m...,CblA beta-lactamase
1,nucleotide_fasta_protein_homolog_model.fasta,gb,HQ845196.1,+,0,861,861,3001109,SHV-52,Klebsiella pneumoniae,861,nucleotide,gb|HQ845196.1|+|0-861|ARO:3001109|SHV-52 [Kleb...,SHV beta-lactamase
2,nucleotide_fasta_protein_homolog_model.fasta,gb,AF028812.1,+,392,887,495,3002867,dfrF,Enterococcus faecalis,495,nucleotide,gb|AF028812.1|+|392-887|ARO:3002867|dfrF [Ente...,trimethoprim resistant dihydrofolate reductase...
3,nucleotide_fasta_protein_homolog_model.fasta,gb,JX017365.1,+,244,1120,876,3001989,CTX-M-130,Escherichia coli,876,nucleotide,gb|JX017365.1|+|244-1120|ARO:3001989|CTX-M-130...,CTX-M beta-lactamase
4,nucleotide_fasta_protein_homolog_model.fasta,gb,JN967644.1,+,0,813,813,3002356,NDM-6,Escherichia coli,813,nucleotide,gb|JN967644.1|+|0-813|ARO:3002356|NDM-6 [Esche...,NDM beta-lactamase


In [11]:
headers_df.to_csv(OUTPUT_HEADERS_WITH_FAMILY_CSV, index=False)
print(f'-> {OUTPUT_HEADERS_WITH_FAMILY_CSV}')

-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/card_reference_headers_with_family.csv


## sec 4

## Gene length distribution for the CARD nucleotide reference set

Mirrors the three-file `Gene_length_distribution*.csv` pattern already used for the
CD-HIT representative-sequence sets under `Results/Genefamily/.../CD/` (see
`AMR_Gene_Family_Distribution.ipynb`, sections 16-17 and 27) — a per (Family, Gene)
summary, the raw per-record long format, and a per-Family summary — applied here to
CARD's own nucleotide reference set (`nucleotide_fasta_protein_homolog_model.fasta`)
instead of per-genome CD-HIT clusters.

Two differences from the CD-derived version:

- **No `Organism` grouping.** In the CD files, `Organism` is the pathogen genome
  each allele was pulled from. In `card_reference_headers_with_family.csv`,
  `organism` is GenBank curation provenance for the reference deposit, not the
  gene's host range (see the note at the top of this notebook) — grouping by it
  would not carry the same meaning, so it is left out here.
- **Nucleotide records only** (`header_type == 'nucleotide'`), since
  `nucleotide_fasta_protein_homolog_model.fasta` is the file this was asked for;
  the protein reference set could be added the same way later if needed.

Self-contained: reads `card_reference_headers_with_family.csv` fresh from disk
(the output of the AMR Gene Family join section above). `accession` is kept as an
extra column beyond the original three-file schema, to trace any outlier length
back to its specific reference record.

In [12]:
REFINED_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details'
INPUT_HEADERS_WITH_FAMILY_CSV = os.path.join(REFINED_DIR, 'card_reference_headers_with_family.csv')
GENE_LENGTH_DISTRIBUTION_CSV = os.path.join(REFINED_DIR, 'card_nucleotide_gene_length_distribution.csv')
GENE_LENGTH_DISTRIBUTION_LONG_CSV = os.path.join(REFINED_DIR, 'card_nucleotide_gene_length_distribution_long.csv')
GENE_LENGTH_DISTRIBUTION_BY_FAMILY_CSV = os.path.join(REFINED_DIR, 'card_nucleotide_gene_length_distribution_by_family.csv')

In [13]:
headers_with_family_df = pd.read_csv(INPUT_HEADERS_WITH_FAMILY_CSV, keep_default_na=False, dtype=str)

nt_gene_length_long_df = (
    headers_with_family_df[headers_with_family_df['header_type'] == 'nucleotide']
    [['amr_gene_family', 'gene_name', 'accession', 'seq_length']]
    .rename(columns={'amr_gene_family': 'AMR Gene Family', 'gene_name': 'Gene', 'seq_length': 'Length'})
    .astype({'Length': int})
    .sort_values(['AMR Gene Family', 'Gene'])
    .reset_index(drop=True)
)
print(f'{len(nt_gene_length_long_df)} nucleotide reference records')
nt_gene_length_long_df.head()

6059 nucleotide reference records


,AMR Gene Family,Gene,accession,Length
0,16S rRNA methyltransferase (A1408),kamB,NG_050561.1,648
1,16S rRNA methyltransferase (A1408),npmA,AB261016.2,660
2,16S rRNA methyltransferase (G1405),armA,GU437214.1,774
3,16S rRNA methyltransferase (G1405),rmtA,AB083212.2,756
4,16S rRNA methyltransferase (G1405),rmtB,AM886293.1,756


### Per (Family, Gene) summary — equivalent of `Gene_length_distribution.csv`

In [14]:
nt_gene_length_per_family_gene_df = (
    nt_gene_length_long_df
    .groupby(['AMR Gene Family', 'Gene'])
    .agg(
        N_Records=('Length', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
        N_Distinct_Lengths=('Length', 'nunique'),
    )
    .reset_index()
    .sort_values(['AMR Gene Family', 'Gene'])
)
print(f'{len(nt_gene_length_per_family_gene_df)} (family, gene) groups')
nt_gene_length_per_family_gene_df.head()

6059 (family, gene) groups


,AMR Gene Family,Gene,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,16S rRNA methyltransferase (A1408),kamB,1,648,648,1
1,16S rRNA methyltransferase (A1408),npmA,1,660,660,1
2,16S rRNA methyltransferase (G1405),armA,1,774,774,1
3,16S rRNA methyltransferase (G1405),rmtA,1,756,756,1
4,16S rRNA methyltransferase (G1405),rmtB,1,756,756,1


### Per Family summary — equivalent of `Gene_length_distribution_3.csv`

In [15]:
nt_gene_length_per_family_df = (
    nt_gene_length_long_df
    .groupby('AMR Gene Family')
    .agg(
        N_Records=('Length', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
        N_Distinct_Lengths=('Length', 'nunique'),
    )
    .reset_index()
    .sort_values('AMR Gene Family')
)
print(f'{len(nt_gene_length_per_family_df)} AMR gene families')
nt_gene_length_per_family_df.head()

406 AMR gene families


,AMR Gene Family,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,16S rRNA methyltransferase (A1408),2,648,660,2
1,16S rRNA methyltransferase (G1405),12,744,846,9
2,AAC(2'),8,537,825,8
3,AAC(3),20,465,900,15
4,AAC(6'),58,402,786,27


In [16]:
nt_gene_length_long_df.to_csv(GENE_LENGTH_DISTRIBUTION_LONG_CSV, index=False)
nt_gene_length_per_family_gene_df.to_csv(GENE_LENGTH_DISTRIBUTION_CSV, index=False)
nt_gene_length_per_family_df.to_csv(GENE_LENGTH_DISTRIBUTION_BY_FAMILY_CSV, index=False)

print(f'-> {GENE_LENGTH_DISTRIBUTION_LONG_CSV}')
print(f'-> {GENE_LENGTH_DISTRIBUTION_CSV}')
print(f'-> {GENE_LENGTH_DISTRIBUTION_BY_FAMILY_CSV}')

-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/card_nucleotide_gene_length_distribution_long.csv
-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/card_nucleotide_gene_length_distribution.csv
-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/card_nucleotide_gene_length_distribution_by_family.csv


## sec 5

## Fixing multi-family (`;`-joined) `AMR Gene Family` values

`aro_index.tsv` encodes many-to-many family membership by joining multiple family
names with `;` in a single cell. This is a genuine CARD convention, not a parsing
error in the file: TolC, for example, really is a member of the ABC, MFS, and RND
efflux-pump families at once (it's the shared outer-membrane channel component),
and `aro_categories.tsv` lists each of those as its own distinct category with its
own ARO Category Accession (`ARO:0010001`/`0010002`/`0010004`).

The join built in the section above treats `AMR Gene Family` as a scalar and
copies these combo-strings through verbatim via `.map()`, which can only assign
one value per key. That silently created phantom "families" that don't exist in
CARD's ontology (e.g. `"ABC...;MFS...;RND..."` as one label) and undercounted the
real families those genes were merged into.

Scope, checked against the actual data: 184 of 6,463 rows in `aro_index.tsv` have
a multi-value `AMR Gene Family` (41 distinct combinations); 272 of 12,118 rows in
`card_reference_headers_with_family.csv` inherited a raw concatenated string as a
result. The pre-existing ESKAPE pipeline (`Gene_length_distribution*.csv`,
`gene_counts.csv`, `AMR_Family_ID_Mapping.csv`, all family FASTA filenames) was
checked and contains no semicolons — it derives family labels from RGI's
per-genome output, not from `aro_index.tsv`, so it is not affected by this.

**Fix**: explode `AMR Gene Family` on `;` before joining, so a gene belonging to
N families produces N rows in the lookup -- a one-row-per-(reference-record,
family) tidy table -- joined with `.merge()` instead of `.map()`, since `.map()`
requires a single scalar per key and cannot represent a one-to-many relationship.

Self-contained: reads `card_reference_headers.csv` and `aro_index.tsv` fresh from
disk, independent of the join section above. Writes to a new output file,
`card_reference_headers_with_family_exploded.csv`, rather than overwriting
`card_reference_headers_with_family.csv`.

In [17]:
REFINED_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details'
INPUT_HEADERS_CSV = os.path.join(REFINED_DIR, 'card_reference_headers.csv')
INPUT_ARO_INDEX_TSV = os.path.join(REFINED_DIR, 'aro_index.tsv')
OUTPUT_HEADERS_WITH_FAMILY_EXPLODED_CSV = os.path.join(REFINED_DIR, 'card_reference_headers_with_family_exploded.csv')

In [18]:
headers_df = pd.read_csv(INPUT_HEADERS_CSV, keep_default_na=False, dtype=str)
aro_index = pd.read_csv(INPUT_ARO_INDEX_TSV, sep='\t', keep_default_na=False, dtype=str)

headers_df['aro_accession_full'] = 'ARO:' + headers_df['aro_accession']

# AMR Gene Family is multi-valued: CARD joins multiple family names with ';' for
# genes that genuinely belong to more than one family (e.g. TolC -> ABC/MFS/RND).
# Explode before building the lookup so each (accession, family) membership is its
# own row, instead of collapsing multi-family genes into one phantom combo string.
family_lookup = aro_index[['ARO Accession', 'AMR Gene Family']].drop_duplicates()
family_lookup['AMR Gene Family'] = family_lookup['AMR Gene Family'].str.split(';')
family_lookup = family_lookup.explode('AMR Gene Family')
family_lookup['AMR Gene Family'] = family_lookup['AMR Gene Family'].str.strip()
family_lookup = family_lookup.drop_duplicates()

conflicting = family_lookup[family_lookup.duplicated(['ARO Accession', 'AMR Gene Family'], keep=False)]
assert conflicting.empty, f'ARO Accession lists the same AMR Gene Family twice:\n{conflicting}'

In [19]:
headers_with_family_exploded_df = headers_df.merge(
    family_lookup.rename(columns={'ARO Accession': 'aro_accession_full', 'AMR Gene Family': 'amr_gene_family'}),
    on='aro_accession_full', how='left'
)

unmatched = headers_with_family_exploded_df[headers_with_family_exploded_df['amr_gene_family'].isna()]
still_combined = headers_with_family_exploded_df[headers_with_family_exploded_df['amr_gene_family'].str.contains(';', na=False)]
print(f'{len(headers_df)} input records -> {len(headers_with_family_exploded_df)} rows after exploding multi-family genes')
print(f'{len(unmatched)} unmatched, {len(still_combined)} still semicolon-joined (should be 0)')

headers_with_family_exploded_df = headers_with_family_exploded_df.drop(columns='aro_accession_full')
headers_with_family_exploded_df.head()

12118 input records -> 12412 rows after exploding multi-family genes
0 unmatched, 0 still semicolon-joined (should be 0)


,source_file,source_db,accession,strand,start,end,cds_length_bp,aro_accession,gene_name,organism,seq_length,header_type,raw_header,amr_gene_family
0,nucleotide_fasta_protein_homolog_model.fasta,gb,GQ343019.1,+,132,1023,891,3002999,CblA-1,mixed culture bacterium AX_gF3SD01_15,891,nucleotide,gb|GQ343019.1|+|132-1023|ARO:3002999|CblA-1 [m...,CblA beta-lactamase
1,nucleotide_fasta_protein_homolog_model.fasta,gb,HQ845196.1,+,0,861,861,3001109,SHV-52,Klebsiella pneumoniae,861,nucleotide,gb|HQ845196.1|+|0-861|ARO:3001109|SHV-52 [Kleb...,SHV beta-lactamase
2,nucleotide_fasta_protein_homolog_model.fasta,gb,AF028812.1,+,392,887,495,3002867,dfrF,Enterococcus faecalis,495,nucleotide,gb|AF028812.1|+|392-887|ARO:3002867|dfrF [Ente...,trimethoprim resistant dihydrofolate reductase...
3,nucleotide_fasta_protein_homolog_model.fasta,gb,JX017365.1,+,244,1120,876,3001989,CTX-M-130,Escherichia coli,876,nucleotide,gb|JX017365.1|+|244-1120|ARO:3001989|CTX-M-130...,CTX-M beta-lactamase
4,nucleotide_fasta_protein_homolog_model.fasta,gb,JN967644.1,+,0,813,813,3002356,NDM-6,Escherichia coli,813,nucleotide,gb|JN967644.1|+|0-813|ARO:3002356|NDM-6 [Esche...,NDM beta-lactamase


In [20]:
headers_with_family_exploded_df.to_csv(OUTPUT_HEADERS_WITH_FAMILY_EXPLODED_CSV, index=False)
print(f'-> {OUTPUT_HEADERS_WITH_FAMILY_EXPLODED_CSV}')

-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/card_reference_headers_with_family_exploded.csv


## sec 6

## Gene length distribution for the CARD nucleotide reference set (multi-family-fixed)

Repeats the "Gene length distribution for the CARD nucleotide reference set"
section above unchanged in method, but sourced from
`card_reference_headers_with_family_exploded.csv` instead of
`card_reference_headers_with_family.csv` — so genes with genuine multi-family
membership (e.g. TolC -> ABC/MFS/RND) are now counted under each real family
they belong to, instead of under one phantom combo-string "family" (see the
"Fixing multi-family..." section above).

Same two differences from the CD-derived pattern as before: no `Organism`
grouping (this table's `organism` is GenBank curation provenance, not host
range), and nucleotide records only. One extra nuance here: because the source
table is exploded, a nucleotide record with N real family memberships now
produces N rows, so row counts below reflect (record x family) pairs, not
distinct reference sequences.

Self-contained: reads `card_reference_headers_with_family_exploded.csv` fresh
from disk. Writes to new output files (`_exploded` suffix) rather than
overwriting the three files the earlier gene-length-distribution section wrote.

In [26]:
REFINED_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details'
OUTPUT_DIR  = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results'
INPUT_HEADERS_WITH_FAMILY_EXPLODED_CSV = os.path.join(REFINED_DIR, 'card_reference_headers_with_family_exploded.csv')
GENE_LENGTH_DISTRIBUTION_EXPLODED_CSV = os.path.join(OUTPUT_DIR, 'card_PROTEIN_length_distribution_exploded.csv')
GENE_LENGTH_DISTRIBUTION_LONG_EXPLODED_CSV = os.path.join(OUTPUT_DIR, 'card_PROTEIN_length_distribution_long_exploded.csv')
GENE_LENGTH_DISTRIBUTION_BY_FAMILY_EXPLODED_CSV = os.path.join(OUTPUT_DIR, 'card_PROTEIN_length_distribution_by_family_exploded.csv')

In [27]:
headers_with_family_exploded_df = pd.read_csv(INPUT_HEADERS_WITH_FAMILY_EXPLODED_CSV, keep_default_na=False, dtype=str)

nt_gene_length_long_exploded_df = (
    headers_with_family_exploded_df[headers_with_family_exploded_df['header_type'] == 'protein']
    [['amr_gene_family', 'gene_name', 'accession', 'seq_length']]
    .rename(columns={'amr_gene_family': 'AMR Gene Family', 'gene_name': 'Gene', 'seq_length': 'Length'})
    .astype({'Length': int})
    .sort_values(['AMR Gene Family', 'Gene'])
    .reset_index(drop=True)
)
print(f'{len(nt_gene_length_long_exploded_df)} nucleotide (record, family) rows '
      f'-- {nt_gene_length_long_exploded_df["accession"].nunique()} distinct accessions')
nt_gene_length_long_exploded_df.head()

6206 nucleotide (record, family) rows -- 6059 distinct accessions


,AMR Gene Family,Gene,accession,Length
0,16S rRNA methyltransferase (A1408),kamB,WP_063964000.1,215
1,16S rRNA methyltransferase (A1408),npmA,BAF80809.1,219
2,16S rRNA methyltransferase (G1405),armA,ADC55560.1,257
3,16S rRNA methyltransferase (G1405),rmtA,BAC20579.1,251
4,16S rRNA methyltransferase (G1405),rmtB,CAP07796.1,251


### Per (Family, Gene) summary — equivalent of `Gene_length_distribution.csv`

In [28]:
nt_gene_length_per_family_gene_exploded_df = (
    nt_gene_length_long_exploded_df
    .groupby(['AMR Gene Family', 'Gene'])
    .agg(
        N_Records=('Length', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
        N_Distinct_Lengths=('Length', 'nunique'),
    )
    .reset_index()
    .sort_values(['AMR Gene Family', 'Gene'])
)
print(f'{len(nt_gene_length_per_family_gene_exploded_df)} (family, gene) groups')
nt_gene_length_per_family_gene_exploded_df.head()

6206 (family, gene) groups


,AMR Gene Family,Gene,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,16S rRNA methyltransferase (A1408),kamB,1,215,215,1
1,16S rRNA methyltransferase (A1408),npmA,1,219,219,1
2,16S rRNA methyltransferase (G1405),armA,1,257,257,1
3,16S rRNA methyltransferase (G1405),rmtA,1,251,251,1
4,16S rRNA methyltransferase (G1405),rmtB,1,251,251,1


### Per Family summary — equivalent of `Gene_length_distribution_3.csv`

In [24]:
nt_gene_length_per_family_exploded_df = (
    nt_gene_length_long_exploded_df
    .groupby('AMR Gene Family')
    .agg(
        N_Records=('Length', 'size'),
        Min_Length=('Length', 'min'),
        Max_Length=('Length', 'max'),
        N_Distinct_Lengths=('Length', 'nunique'),
    )
    .reset_index()
    .sort_values('AMR Gene Family')
)
print(f'{len(nt_gene_length_per_family_exploded_df)} AMR gene families (was 406 before the multi-family fix)')
nt_gene_length_per_family_exploded_df.head()

396 AMR gene families (was 406 before the multi-family fix)


,AMR Gene Family,N_Records,Min_Length,Max_Length,N_Distinct_Lengths
0,16S rRNA methyltransferase (A1408),2,648,660,2
1,16S rRNA methyltransferase (G1405),12,744,846,9
2,AAC(2'),8,537,825,8
3,AAC(3),20,465,900,15
4,AAC(6'),68,402,786,27


In [29]:
nt_gene_length_long_exploded_df.to_csv(GENE_LENGTH_DISTRIBUTION_LONG_EXPLODED_CSV, index=False)
nt_gene_length_per_family_gene_exploded_df.to_csv(GENE_LENGTH_DISTRIBUTION_EXPLODED_CSV, index=False)
nt_gene_length_per_family_exploded_df.to_csv(GENE_LENGTH_DISTRIBUTION_BY_FAMILY_EXPLODED_CSV, index=False)

print(f'-> {GENE_LENGTH_DISTRIBUTION_LONG_EXPLODED_CSV}')
print(f'-> {GENE_LENGTH_DISTRIBUTION_EXPLODED_CSV}')
print(f'-> {GENE_LENGTH_DISTRIBUTION_BY_FAMILY_EXPLODED_CSV}')

-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/card_PROTEIN_length_distribution_long_exploded.csv
-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/card_PROTEIN_length_distribution_exploded.csv
-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/card_PROTEIN_length_distribution_by_family_exploded.csv


## sec 7

## Splitting the CARD protein reference set into per-family FASTA files

Organizes `protein_fasta_protein_homolog_model.fasta` into one multi-FASTA file
per AMR Gene Family (396 files), with shortened headers for downstream tools
(e.g. pfeature) that don't handle CARD's full pipe-delimited headers well.

Inputs:
- `protein_fasta_protein_homolog_model.fasta` -- the source sequences.
- `Final_Results/gene_family_match.csv` -- `family_name,short_code` (e.g.
  `CblA beta-lactamase,f1`), no header row.
- `Final_Results/card_PROTEIN_length_distribution_long_exploded.csv` -- one row
  per (accession, Gene, AMR Gene Family) membership, already exploded so a
  multi-family protein appears once per real family it belongs to (see the
  "Fixing multi-family..." section above).

Checked before writing this: the two input files' family-name sets are
identical (396 == 396, no gaps either direction), and the long-exploded CSV's
`(accession, Gene)` keys match every one of the 6,059 records in the source
FASTA exactly -- so joining the freshly-parsed FASTA against it on
`(accession, Gene)` is fully reliable.

Header format: `>{short_code}_{aro_accession}`, e.g. CblA-1 (family CblA
beta-lactamase -> f1, ARO:3002999) becomes `>f1_3002999` -- reusing the ARO
accession already embedded in the original header rather than the GenBank
accession, and dropping the organism/gene-name text entirely.

A protein record belonging to N real families (post multi-family-fix) is
written into N separate family files, once each -- this is intentional,
matching how the multi-family fix works everywhere else in this notebook.

One filename fixup: the family `Serine/threonine kinases` contains a literal
`/`, which would be read as a path separator; only the *filename* replaces `/`
with `_` (-> `Serine_threonine kinases.fasta`) -- the family name inside the
data (headers routed to that file, the CSVs) is untouched.

Self-contained: reads all three inputs fresh from disk, independent of the
notebook's earlier `parse_card_header`/`read_fasta_records` (re-declares a lean
protein-only version here). Writes to a new folder,
`Final_Results/Protein_Family_Fastas/`, without touching any existing file.

In [30]:
CARD_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data'
FINAL_RESULTS_DIR = os.path.join(CARD_DIR, 'refined_details', 'Final_Results')

PROTEIN_FASTA_PATH = os.path.join(CARD_DIR, 'protein_fasta_protein_homolog_model.fasta')
GENE_FAMILY_MATCH_CSV = os.path.join(FINAL_RESULTS_DIR, 'gene_family_match.csv')
PROTEIN_LENGTH_DISTRIBUTION_LONG_CSV = os.path.join(FINAL_RESULTS_DIR, 'card_PROTEIN_length_distribution_long_exploded.csv')
FAMILY_FASTA_OUTPUT_DIR = os.path.join(FINAL_RESULTS_DIR, 'Protein_Family_Fastas')

In [31]:
PAT_PROT = re.compile(
    r'^(?P<source_db>[^|]+)\|(?P<accession>[^|]+)\|ARO:(?P<aro_accession>\d+)\|(?P<rest>.*)$'
)
PAT_ORG = re.compile(r'^(?P<gene_name>.*?)\s*\[(?P<organism>[^\]]+)\]\s*$')

def parse_protein_header(header):
    """Return {'accession', 'aro_accession', 'gene_name'} or None if unmatched."""
    match = PAT_PROT.match(header)
    if not match:
        return None
    d = match.groupdict()
    rest = d.pop('rest')
    org_match = PAT_ORG.match(rest)
    d['gene_name'] = org_match.group('gene_name') if org_match else rest.strip()
    return d

def read_fasta_records(path):
    """Yield (header, sequence) pairs. Header excludes the leading '>'."""
    header, seq = None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(seq)
                header, seq = line[1:].strip(), []
            else:
                seq.append(line.strip())
        if header is not None:
            yield header, ''.join(seq)

In [32]:
fam_match_df = pd.read_csv(GENE_FAMILY_MATCH_CSV, header=None, names=['family', 'short_code'], dtype=str)
family_short_code = dict(zip(fam_match_df['family'].str.strip(), fam_match_df['short_code'].str.strip()))

long_df = pd.read_csv(PROTEIN_LENGTH_DISTRIBUTION_LONG_CSV, dtype=str)
family_membership = (
    long_df.groupby(['accession', 'Gene'])['AMR Gene Family']
    .apply(list)
    .to_dict()
)

print(f'{len(family_short_code)} families with a short code')
print(f'{len(family_membership)} (accession, gene) keys with known family membership')

396 families with a short code
6059 (accession, gene) keys with known family membership


In [33]:
os.makedirs(FAMILY_FASTA_OUTPUT_DIR, exist_ok=True)

open_files = {}
n_records, n_written, n_no_family_match = 0, 0, 0
unmatched = []

for header, seq in read_fasta_records(PROTEIN_FASTA_PATH):
    record = parse_protein_header(header)
    if record is None:
        continue
    n_records += 1

    families = family_membership.get((record['accession'], record['gene_name']))
    if not families:
        n_no_family_match += 1
        unmatched.append((record['accession'], record['gene_name']))
        continue

    for family in families:
        short_code = family_short_code[family]
        safe_filename = family.replace('/', '_') + '.fasta'
        fasta_path = os.path.join(FAMILY_FASTA_OUTPUT_DIR, safe_filename)
        if fasta_path not in open_files:
            open_files[fasta_path] = open(fasta_path, 'w')
        open_files[fasta_path].write(f">{short_code}_{record['aro_accession']}\n{seq}\n")
        n_written += 1

for f in open_files.values():
    f.close()

print(f'{n_records} protein records read')
print(f'{n_written} (sequence, family) records written across {len(open_files)} family FASTA files')
print(f'{n_no_family_match} records with no family match (should be 0)')
if unmatched:
    print('unmatched examples:', unmatched[:5])

6059 protein records read
6206 (sequence, family) records written across 396 family FASTA files
0 records with no family match (should be 0)


## Collapsing the 396 family FASTAs back to the individual protein records

Parses the 396 per-family FASTA files written by the section above back down to
one deduplicated multi-FASTA holding every distinct protein record exactly once
-- the input set for pairwise sequence alignment.

The 396 family files hold **6,206** headers but only **6,059** distinct protein
records. The 147-record gap is not an error: it is the 136 proteins that
genuinely belong to more than one gene family (126 in 2 families, 9 in 3, 1 in
4 -- 126x1 + 9x2 + 1x3 = 147 extra copies), each written once per family by the
previous section. Checked before writing this: every duplicate copy of a record
carries a byte-identical sequence, and no two different ARO accessions share a
sequence -- so the ARO accession is a clean unique key for all 6,059 records.

Rather than discard the extra family memberships when collapsing, the header
keeps **all** of them:

| record | header |
|---|---|
| single-family (5,923 records) | `>f1_3002999` |
| in families f37 + f165 (136 records) | `>f37_f165_3002547` |

Short codes are ordered numerically (`f37` before `f165`) -- plain string
sorting would invert them, since `'f165' < 'f37'` lexicographically. Records are
emitted in ascending ARO-accession order so the output is stable across runs.

Self-contained: reads the 396 family FASTAs fresh from disk and writes one new
file in a new folder, without touching anything that already exists.

In [34]:
FINAL_RESULTS_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results'
INPUT_FAMILY_FASTA_DIR = os.path.join(FINAL_RESULTS_DIR, 'Protein_Family_Fastas')
INDIVIDUAL_RECORDS_DIR = os.path.join(FINAL_RESULTS_DIR, 'Protein_Individual_Records')
OUTPUT_INDIVIDUAL_RECORDS_FASTA = os.path.join(INDIVIDUAL_RECORDS_DIR, 'card_protein_individual_records.fasta')

In [35]:
from collections import defaultdict


def read_fasta_records(path):
    """Yield (header, sequence) pairs. Header excludes the leading '>'."""
    header, seq = None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(seq)
                header, seq = line[1:].strip(), []
            else:
                seq.append(line.strip())
        if header is not None:
            yield header, ''.join(seq)


def short_code_order(short_code):
    """'f37' -> 37, so f37 sorts before f165 ('f165' < 'f37' as plain strings)."""
    return int(short_code[1:])

In [36]:
family_fasta_files = sorted(f for f in os.listdir(INPUT_FAMILY_FASTA_DIR) if f.endswith('.fasta'))

record_seq = {}                           # aro_accession -> sequence
record_short_codes = defaultdict(set)     # aro_accession -> {family short codes}
n_headers = 0

for filename in family_fasta_files:
    for header, seq in read_fasta_records(os.path.join(INPUT_FAMILY_FASTA_DIR, filename)):
        n_headers += 1
        short_code, aro_accession = header.rsplit('_', 1)
        record_short_codes[aro_accession].add(short_code)
        # the same record written into N family files must carry the same sequence
        assert record_seq.get(aro_accession, seq) == seq, f'sequence mismatch for ARO {aro_accession}'
        record_seq[aro_accession] = seq

n_multi_family = sum(1 for codes in record_short_codes.values() if len(codes) > 1)
print(f'{len(family_fasta_files)} family FASTA files read')
print(f'{n_headers} headers -> {len(record_seq)} distinct protein records')
print(f'{n_headers - len(record_seq)} duplicate copies collapsed, from {n_multi_family} multi-family records')

396 family FASTA files read
6206 headers -> 6059 distinct protein records
147 duplicate copies collapsed, from 136 multi-family records


In [37]:
os.makedirs(INDIVIDUAL_RECORDS_DIR, exist_ok=True)

with open(OUTPUT_INDIVIDUAL_RECORDS_FASTA, 'w') as out:
    for aro_accession in sorted(record_seq, key=int):
        codes = '_'.join(sorted(record_short_codes[aro_accession], key=short_code_order))
        out.write(f'>{codes}_{aro_accession}\n{record_seq[aro_accession]}\n')

print(f'{len(record_seq)} records -> {OUTPUT_INDIVIDUAL_RECORDS_FASTA}')

print('\nexample multi-family headers:')
multi = {a: c for a, c in record_short_codes.items() if len(c) > 1}
for aro_accession in sorted(multi, key=int)[:3]:
    print('  >' + '_'.join(sorted(multi[aro_accession], key=short_code_order)) + '_' + aro_accession)

6059 records -> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Protein_Individual_Records/card_protein_individual_records.fasta

example multi-family headers:
  >f63_f64_3000005
  >f63_f64_3000010
  >f63_f64_3000013


## All-vs-all pairwise alignment of the 6,059 protein records

Aligns every one of the 6,059 deduplicated protein records against every other
and writes two square matrices -- percent similarity and percent identity --
labelled with the FASTA headers down the first column and across the first row.

Method: Needleman-Wunsch (**global**) with BLOSUM62, gap open -11 / extend -1,
and terminal gaps unpenalised, which is EMBOSS `needle`'s default behaviour
(`endweight=N`). Both percentages use the **alignment length including gap
columns** as the denominator, also the EMBOSS convention:

| metric | numerator |
|---|---|
| % similarity | `positives` -- identities **plus** positive-scoring substitutions |
| % identity | `identities` -- exact matches only |

Identity is therefore always <= similarity. Both come out of the same alignment
pass, so producing the second matrix costs no extra compute.

This is **18,352,711 pairwise alignments** (~1.9 trillion DP cells) and takes
roughly an hour and a half across 12 workers, so it runs as a standalone script
rather than a notebook cell -- a multi-hour cell is fragile, and the worker
processes would be competing with this kernel for memory. Progress is
checkpointed per matrix row, so an interrupted run resumes rather than
restarting.

```bash
cd /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family

# smoke test on the first 200 records (~13 s, isolated *_test200.* outputs)
python3 card_pairwise_alignment.py --max-records 200

# the real run, in the background
nohup python3 card_pairwise_alignment.py --workers 12 > pairwise_alignment.log 2>&1 &
tail -f pairwise_alignment.log
```

Outputs land in `Final_Results/Pairwise_Alignment/`: the two labelled CSVs
(~250 MB each) plus `.npy` copies, which are far faster to reload than parsing
36.7 M CSV cells. The cell below reads the `.npy` back once the run has
finished.

In [ ]:
PAIRWISE_DIR = os.path.join(FINAL_RESULTS_DIR, 'Pairwise_Alignment')
SIMILARITY_NPY = os.path.join(PAIRWISE_DIR, 'pairwise_similarity.npy')
IDENTITY_NPY = os.path.join(PAIRWISE_DIR, 'pairwise_identity.npy')

import numpy as np

record_names = [line[1:].strip() for line in open(OUTPUT_INDIVIDUAL_RECORDS_FASTA)
                if line.startswith('>')]
similarity_matrix = np.load(SIMILARITY_NPY, mmap_mode='r')
identity_matrix = np.load(IDENTITY_NPY, mmap_mode='r')

print(f'{similarity_matrix.shape[0]} x {similarity_matrix.shape[1]} matrix, '
      f'{len(record_names)} headers')
print(f'diagonal all 100: {bool(np.all(np.diag(similarity_matrix) == 100.0))}')
print(f'value range     : {similarity_matrix.min():.2f} - {similarity_matrix.max():.2f}')

# labelled top-left corner, as a readable preview of the full matrix
pd.DataFrame(similarity_matrix[:6, :6], index=record_names[:6], columns=record_names[:6])

## Mapping AMR Gene Family to Drug Class from `aro_index.tsv`

Builds the family <-> drug class mapping from CARD's own reference index rather
than from RGI per-genome output.

This mirrors the idiom in `ARG_Family_classification.ipynb` (sections
`## Frequency of AMR gene families` and `## Map the Drug class to gene family`):
split the semicolon-joined cell, `explode()` to one value per row, strip
whitespace. One deliberate difference -- **both** columns are exploded here.

In `aro_index.tsv` both fields are multi-valued, `Drug Class` far more so:

| | `AMR Gene Family` | `Drug Class` |
|---|---|---|
| rows containing `;` | 184 / 6,463 | 4,096 / 6,463 |
| distinct raw strings | 552 | 150 |
| of which are combo-strings | 41 | 110 |
| real values after exploding | **539** | **53** |
| real values that *only* ever appear inside a combo | 28 | 13 |

That last row is why exploding one side only is not good enough. The
RGI-derived version explodes just `Drug Class`, leaving `AMR Gene Family` as raw
combo-strings -- the same defect fixed earlier in this notebook (see "Fixing
multi-family (`;`-joined) `AMR Gene Family` values"). Left unexploded, the
family column carries 41 phantom "families" that exist nowhere in CARD's
ontology, while 28 real families stay invisible because they are only ever
listed bundled with another.

Result: **981 unique (family, drug class) pairs**. 217 of the 539 families map
to more than one drug class -- MFS efflux pump spans 28, ABC 24, RND 21 -- the
efflux-pump polyspecificity described in
`Results/DRUG_class_family_mapping/Gene_Family_Alignment_and_Drug_Class_Discussion.md`.

**What a row means here.** This is a catalogue of CARD *models*, not a frequency
over observed genes. The RGI-derived mapping in
`Results/DRUG_class_family_mapping/` is occurrence-weighted -- 31,489 rows for
*A. baumannii* alone, one per hit across its genomes -- whereas this table has
exactly one row per distinct family/drug-class relationship.

Scope is all 539 families in `aro_index.tsv`, covering every CARD model type --
wider than the 396 protein-homolog families used elsewhere in this notebook.

Self-contained: reads `aro_index.tsv` fresh from disk and writes one new file,
touching nothing that already exists.

In [3]:
REFINED_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details'
FINAL_RESULTS_DIR = os.path.join(REFINED_DIR, 'Final_Results')
INPUT_ARO_INDEX_TSV = os.path.join(REFINED_DIR, 'aro_index.tsv')
OUTPUT_FAMILY_DRUG_CLASS_CSV = os.path.join(FINAL_RESULTS_DIR, 'card_family_drug_class_mapping.csv')

In [39]:
aro_index = pd.read_csv(INPUT_ARO_INDEX_TSV, sep='\t', keep_default_na=False, dtype=str)

# Both columns are multi-valued -- CARD joins several values with ';' in a single
# cell, Drug Class in 4,096 of 6,463 rows -- so BOTH get exploded. The RGI-derived
# version in ARG_Family_classification.ipynb explodes only Drug Class, which leaves
# 41 phantom combo-strings in the family column and hides 28 real families.
family_drug_class = aro_index[['AMR Gene Family', 'Drug Class']].copy()

family_drug_class['AMR Gene Family'] = family_drug_class['AMR Gene Family'].str.split(';')
family_drug_class = family_drug_class.explode('AMR Gene Family')
family_drug_class['Drug Class'] = family_drug_class['Drug Class'].str.split(';')
family_drug_class = family_drug_class.explode('Drug Class')

for column in ('AMR Gene Family', 'Drug Class'):
    family_drug_class[column] = family_drug_class[column].str.strip()

family_drug_class = family_drug_class[(family_drug_class['AMR Gene Family'] != '')
                                      & (family_drug_class['Drug Class'] != '')]
family_drug_class = (family_drug_class
                     .drop_duplicates()
                     .sort_values(['AMR Gene Family', 'Drug Class'])
                     .reset_index(drop=True))

still_combined = family_drug_class[family_drug_class['AMR Gene Family'].str.contains(';')
                                   | family_drug_class['Drug Class'].str.contains(';')]
print(f'{len(family_drug_class)} unique (family, drug class) pairs')
print(f'{family_drug_class["AMR Gene Family"].nunique()} families, '
      f'{family_drug_class["Drug Class"].nunique()} drug classes')
print(f'{len(still_combined)} rows still semicolon-joined (should be 0)')
family_drug_class.head()

981 unique (family, drug class) pairs
539 families, 53 drug classes
0 rows still semicolon-joined (should be 0)


,AMR Gene Family,Drug Class
0,16S rRNA methyltransferase (A1408),aminoglycoside antibiotic
1,16S rRNA methyltransferase (G1405),aminoglycoside antibiotic
2,16S rRNA with mutation conferring resistance t...,pactamycin-like antibiotic
3,16S rRNA with mutation conferring resistance t...,tetracycline antibiotic
4,16s rRNA with mutation conferring resistance t...,aminoglycoside antibiotic


In [40]:
family_drug_class.to_csv(OUTPUT_FAMILY_DRUG_CLASS_CSV, index=False)
print(f'-> {OUTPUT_FAMILY_DRUG_CLASS_CSV}')

-> /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/card_family_drug_class_mapping.csv


## Create nucleotide and protein FASTA files by CARD drug class

This standalone section uses fresh configuration and does not depend on variables from earlier notebook sections. A sequence assigned to several CARD drug classes is written once to each applicable drug-class FASTA.


In [4]:
# Fresh configuration for drug-class FASTA extraction.
from pathlib import Path

CARD_DATA_DIR = Path('/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data')
DRUG_CLASS_ARO_INDEX_TSV = CARD_DATA_DIR / 'refined_details' / 'aro_index.tsv'
DRUG_CLASS_NUCLEOTIDE_FASTA = CARD_DATA_DIR / 'nucleotide_fasta_protein_homolog_model.fasta'
DRUG_CLASS_PROTEIN_FASTA = CARD_DATA_DIR / 'protein_fasta_protein_homolog_model.fasta'
DRUG_CLASS_OUTPUT_ROOT = CARD_DATA_DIR / 'refined_details' / 'Final_Results' / 'Drug_Class_Fastas'
DRUG_CLASS_MANIFEST_CSV = CARD_DATA_DIR / 'refined_details' / 'Final_Results' / 'card_sequence_drug_class_manifest.csv'


In [5]:
import csv
import re
from collections import defaultdict

DRUG_CLASS_NT_HEADER = re.compile(
    r'^(?P<source_db>[^|]+)\|(?P<accession>[^|]+)\|(?P<strand>[+-])\|'
    r'(?P<start>\d+)-(?P<end>\d+)\|ARO:(?P<aro_accession>\d+)\|(?P<description>.*)$'
)
DRUG_CLASS_PROTEIN_HEADER = re.compile(
    r'^(?P<source_db>[^|]+)\|(?P<accession>[^|]+)\|ARO:(?P<aro_accession>\d+)\|(?P<description>.*)$'
)
DRUG_CLASS_ORGANISM = re.compile(r'^(?P<gene_name>.*?)\s*\[(?P<organism>[^\]]+)\]\s*$')


def read_drug_class_fasta(path):
    """Yield (header, sequence) pairs; header excludes the leading >."""
    header, sequence_lines = None, []
    with Path(path).open(encoding='utf-8') as handle:
        for line in handle:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(sequence_lines)
                header, sequence_lines = line[1:].strip(), []
            else:
                sequence_lines.append(line.strip())
        if header is not None:
            yield header, ''.join(sequence_lines)


def parse_drug_class_header(header, sequence_type):
    """Parse a CARD homolog-model header into traceable sequence metadata."""
    pattern = DRUG_CLASS_NT_HEADER if sequence_type == 'nucleotide' else DRUG_CLASS_PROTEIN_HEADER
    match = pattern.match(header)
    if match is None:
        return None

    record = match.groupdict()
    organism_match = DRUG_CLASS_ORGANISM.match(record['description'])
    record['gene_name'] = organism_match.group('gene_name').strip() if organism_match else record['description'].strip()
    record['organism'] = organism_match.group('organism').strip() if organism_match else ''
    record['sequence_type'] = sequence_type
    record['raw_header'] = header
    return record


def drug_class_filename(drug_class):
    """Return a portable, deterministic filename for one CARD drug-class label."""
    safe_name = re.sub(r'[^A-Za-z0-9._-]+', '_', drug_class.strip())
    return safe_name.strip('._') + '.fasta'


def load_aro_to_drug_classes(aro_index_path):
    """Return ARO accession -> sorted unique CARD drug classes."""
    aro_to_classes = defaultdict(set)
    with Path(aro_index_path).open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle, delimiter='\t')
        required_columns = {'ARO Accession', 'Drug Class'}
        missing_columns = required_columns.difference(reader.fieldnames or [])
        if missing_columns:
            raise ValueError(f'aro_index.tsv is missing required columns: {sorted(missing_columns)}')
        for row in reader:
            aro_accession = (row['ARO Accession'] or '').strip()
            for drug_class in (row['Drug Class'] or '').split(';'):
                drug_class = drug_class.strip()
                if aro_accession and drug_class:
                    aro_to_classes[aro_accession].add(drug_class)
    return {aro: sorted(classes, key=str.casefold) for aro, classes in aro_to_classes.items()}


In [6]:
# Run this cell only when you are ready to create the drug-class FASTAs and manifest.
aro_to_drug_classes = load_aro_to_drug_classes(DRUG_CLASS_ARO_INDEX_TSV)
(DRUG_CLASS_OUTPUT_ROOT / 'nucleotide').mkdir(parents=True, exist_ok=True)
(DRUG_CLASS_OUTPUT_ROOT / 'protein').mkdir(parents=True, exist_ok=True)
DRUG_CLASS_MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)

manifest_columns = [
    'sequence_type', 'source_db', 'accession', 'aro_accession', 'gene_name',
    'organism', 'drug_class', 'output_fasta', 'sequence_length', 'raw_header',
]
manifest_rows = []
unparsed_headers = []
unmatched_records = []
input_records = 0
assignments_written = 0
output_files = set()

for sequence_type, fasta_path in {
    'nucleotide': DRUG_CLASS_NUCLEOTIDE_FASTA,
    'protein': DRUG_CLASS_PROTEIN_FASTA,
}.items():
    open_files = {}
    try:
        for raw_header, sequence in read_drug_class_fasta(fasta_path):
            input_records += 1
            record = parse_drug_class_header(raw_header, sequence_type)
            if record is None:
                unparsed_headers.append((sequence_type, raw_header))
                continue

            aro_full = 'ARO:' + record['aro_accession']
            drug_classes = aro_to_drug_classes.get(aro_full, [])
            if not drug_classes:
                unmatched_records.append((sequence_type, record['accession'], aro_full, record['gene_name']))
                continue

            for drug_class in drug_classes:
                output_path = DRUG_CLASS_OUTPUT_ROOT / sequence_type / drug_class_filename(drug_class)
                if output_path not in open_files:
                    open_files[output_path] = output_path.open('w', encoding='utf-8')
                # Keep ARO accession and the original CARD annotation for full provenance.
                open_files[output_path].write(f">ARO:{record['aro_accession']}|{raw_header}\n{sequence}\n")

                manifest_rows.append({
                    'sequence_type': sequence_type,
                    'source_db': record['source_db'],
                    'accession': record['accession'],
                    'aro_accession': aro_full,
                    'gene_name': record['gene_name'],
                    'organism': record['organism'],
                    'drug_class': drug_class,
                    'output_fasta': str(output_path),
                    'sequence_length': len(sequence),
                    'raw_header': raw_header,
                })
                assignments_written += 1
                output_files.add(output_path)
    finally:
        for handle in open_files.values():
            handle.close()

with DRUG_CLASS_MANIFEST_CSV.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=manifest_columns)
    writer.writeheader()
    writer.writerows(manifest_rows)

unique_drug_classes = {row['drug_class'] for row in manifest_rows}
multi_class_assignments = assignments_written - (input_records - len(unparsed_headers) - len(unmatched_records))
print(f'Input FASTA records: {input_records}')
print(f'Unparseable headers: {len(unparsed_headers)}')
print(f'No drug-class match: {len(unmatched_records)}')
print(f'Sequence-to-class assignments written: {assignments_written}')
print(f'Additional multi-class assignments: {multi_class_assignments}')
print(f'Unique drug classes represented: {len(unique_drug_classes)}')
print(f'Drug-class FASTA files written: {len(output_files)}')
print(f'Manifest: {DRUG_CLASS_MANIFEST_CSV}')
if unparsed_headers:
    print('Unparseable-header examples:', unparsed_headers[:5])
if unmatched_records:
    print('Unmatched-record examples:', unmatched_records[:5])


Input FASTA records: 12118
Unparseable headers: 0
No drug-class match: 0
Sequence-to-class assignments written: 25826
Additional multi-class assignments: 13708
Unique drug classes represented: 39
Drug-class FASTA files written: 78
Manifest: /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/card_sequence_drug_class_manifest.csv
